# ESP32 로봇 강아지 4방향 보행

## 필요한 펌웨어

이 노트북은 `esp32-robot-dog-code-master/tools/servoInitPose/servoInitPose.ino` 펌웨어와 함께 사용합니다. 이 파일의 WALK 명령 지원 버전을 Arduino IDE에서 ESP32에 업로드하세요. 이전 펌웨어를 덮어쓰게 됩니다. Arduino IDE에서 보드 `ESP32 Dev Module`, 업로드 속도는 기본 설정을 사용하고, 노트북 시리얼 통신 속도는 `115200`입니다. 펌웨어는 ESP32 Arduino core 3.x의 내장 LEDC API를 사용하며 core 2.x에서는 기존 `ESP32_ISR_Servo` 분기를 사용합니다. 업로드가 끝나면 Arduino IDE의 시리얼 모니터/플로터를 닫은 뒤 노트북에서 COM 포트를 연결하세요.

펌웨어는 `HELLO`, `STATUS`, `WALK FORWARD`, `WALK BACKWARD`, `WALK LEFT`, `WALK RIGHT`, `WALK STOP` 명령을 지원해야 합니다. 다른 스케치가 올라가 있으면 WALK 버튼은 동작하지 않습니다.

## 사용 순서

1. 로봇을 안전하게 고정하고 발을 바닥에서 띄운 상태로 시작합니다. 전원 차단 스위치를 손 닿는 곳에 두고, 전원과 배선을 확인하세요.
2. 아래 셀을 실행하고 `COM14 연결`을 누릅니다. 연결할 때 펌웨어 식별 후 STOP 응답을 확인하기 전까지 움직임 활성화를 허용하지 않습니다. 다른 COM 포트라면 코드의 `PORT` 값을 바꾸세요.
3. `움직임 활성화`를 누른 다음 방향 버튼을 누르세요. 같은 방향을 다시 누르면 정지하고, 다른 방향을 누르면 방향을 바꿉니다. STOP 응답이 없으면 연결을 유지해 정지를 재시도할 수 있게 하며, 로봇 상태가 불명확하면 즉시 전원을 차단하세요. `연결 해제`는 STOP 응답을 확인한 뒤 포트를 닫습니다.

## 보행 프로토타입 안전 주의

현재 보행은 작은 각도 범위의 개방 루프 프로토타입입니다. 펌웨어는 대각선 다리 쌍을 번갈아 움직이며, 발 접촉 센서나 자세 균형 제어가 없습니다. STOP 응답은 펌웨어가 보행 상태를 해제하고 중립 목표각을 설정했다는 뜻이며, 서보가 물리적으로 멈췄거나 중립 위치에 도달했다는 보장은 아닙니다. 실제 바닥 주행이 안전하거나 정상 동작한다고 보장되지 않습니다. 먼저 발을 띄운 상태에서 아주 짧게 시험해 다리 방향과 기구 간섭을 확인하세요. 끼임, 충돌, 떨림, 비정상 소음이 있으면 긴급 정지 후 서보 전원을 끄세요. 각도 제한에만 의존하지 마세요.

## 하트비트 (자동 정지)

펌웨어는 `CAPS`에 `LEASE1`이 있는 버전이어야 합니다. 걷는 동안 이 노트북은 0.25초마다 `HB`를 보냅니다. 1초 동안 `HB`가 오지 않으면 (커널 중지·재시작, USB 분리, 노트북 종료) 로봇은 스스로 멈추고 비활성화됩니다. 하트비트가 실패하면 버튼이 비활성화되니 `긴급 정지`로 상태를 확인한 뒤 다시 활성화하세요. 정지하면 모든 관절이 중립 자세로 천천히 돌아갑니다.


In [1]:
import html
import threading
import time

import ipywidgets as widgets
import serial
from IPython.display import clear_output, display

PORT = "COM14"
BAUD = 115200
esp = None
movement_enabled = False
active_direction = None
motion_state_unknown = True
# One command/reply at a time: the heartbeat thread shares the port.
serial_lock = threading.RLock()
HEARTBEAT_INTERVAL_S = 0.25
heartbeat_thread = None

port_status = widgets.HTML(value="Not connected. Close Arduino Serial Monitor, then connect the board.")
connect_button = widgets.Button(description="COM14 연결", button_style="info")
disconnect_button = widgets.Button(description="연결 해제")
arm_button = widgets.Button(description="움직임 활성화", button_style="warning")
disarm_button = widgets.Button(description="비활성화", disabled=True)
walk_status = widgets.HTML(value="정지")
walk_buttons = {
    "FORWARD": widgets.Button(description="앞으로", button_style="success", disabled=True),
    "BACKWARD": widgets.Button(description="뒤로", button_style="success", disabled=True),
    "LEFT": widgets.Button(description="왼쪽", button_style="success", disabled=True),
    "RIGHT": widgets.Button(description="오른쪽", button_style="success", disabled=True),
}
stop_button = widgets.Button(description="긴급 정지", button_style="danger", disabled=True)
ui_output = widgets.Output()

def open_serial_without_reset(port, baud):
    # Keep DTR/RTS deasserted while opening so the ESP32 auto-reset circuit
    # does not reboot the board (a reboot drops all servo pulses).
    port_handle = serial.Serial()
    port_handle.port = port
    port_handle.baudrate = baud
    port_handle.timeout = 0.25
    port_handle.write_timeout = 2
    port_handle.dsrdtr = False
    port_handle.rtscts = False
    port_handle.dtr = False
    port_handle.rts = False
    port_handle.open()
    return port_handle

def send_command(command, require_motion=False):
    if esp is None or not esp.is_open:
        raise RuntimeError("ESP32가 연결되지 않았습니다")
    if require_motion and not movement_enabled:
        raise RuntimeError("먼저 움직임을 활성화하세요")
    prefix = {
        "STATUS": "STATUS ",
        "CAPS": "CAPS ",
        "WALK": "OK WALK ",
        "HB": "OK HB",
        "SETTLED": "SETTLED ",
        "RESET_REASON": "RESET_REASON ",
    }.get(command.split(maxsplit=1)[0])
    with serial_lock:
        # Drop late replies to an earlier timed-out command so they are not
        # taken as the answer to this one.
        esp.reset_input_buffer()
        esp.write((command + "\r\n").encode("ascii"))
        deadline = time.monotonic() + 3
        while time.monotonic() < deadline:
            reply = esp.readline().decode("ascii", errors="replace").strip()
            if not reply or reply.startswith("EVENT "):
                continue
            if reply.startswith(("ERR", "ERROR:")):
                raise RuntimeError(reply)
            if prefix is None or reply.startswith(prefix):
                return reply
    raise TimeoutError(f"{command} 응답을 받지 못했습니다")

def wait_for_settled(timeout_s=6.0):
    # Close the port only after the servos reached their targets and the pose is
    # saved in flash. If the ESP32 resets later (USB driver, brownout), it then
    # restarts from the real servo positions instead of jumping at full speed.
    deadline = time.monotonic() + timeout_s
    while time.monotonic() < deadline:
        if send_command("SETTLED") == "SETTLED 1":
            return True
        time.sleep(0.2)
    return False

def heartbeat_loop(connection):
    # The firmware stops and disarms when no HB arrives for 1 s (kernel crash,
    # USB unplugged, notebook closed). Runs only while a direction is active.
    global movement_enabled, active_direction, motion_state_unknown
    while esp is connection and active_direction is not None:
        try:
            send_command("HB")
        except (RuntimeError, TimeoutError, OSError) as exc:
            if esp is connection and active_direction is not None:
                movement_enabled = False
                active_direction = None
                motion_state_unknown = True
                walk_status.value = f"하트비트 실패 - 로봇은 1초 안에 스스로 정지합니다. STOP으로 상태를 확인하세요 ({html.escape(str(exc))})"
                update_controls()
            return
        time.sleep(HEARTBEAT_INTERVAL_S)

def start_heartbeat():
    global heartbeat_thread
    if heartbeat_thread is not None and heartbeat_thread.is_alive():
        return
    heartbeat_thread = threading.Thread(target=heartbeat_loop, args=(esp,), daemon=True)
    heartbeat_thread.start()

def update_controls():
    connected = esp is not None and esp.is_open
    connect_button.disabled = connected
    disconnect_button.disabled = not connected
    arm_button.disabled = not connected or movement_enabled or motion_state_unknown
    disarm_button.disabled = not movement_enabled
    for direction, button in walk_buttons.items():
        button.disabled = not movement_enabled
        button.button_style = "warning" if direction == active_direction else "success"
    stop_button.disabled = not connected

def show_result(action):
    with ui_output:
        clear_output(wait=True)
        try:
            result = action()
        except (RuntimeError, ValueError, TypeError, TimeoutError, OSError) as exc:
            print(f"오류: {exc}")
        else:
            if result:
                print(result)

def connect_board(_):
    global esp, motion_state_unknown
    connect_button.disabled = True
    motion_state_unknown = True
    port_status.value = f"Connecting to {html.escape(PORT)}..."
    candidate = None
    try:
        candidate = open_serial_without_reset(PORT, BAUD)
        deadline = time.monotonic() + 12
        next_hello = 0.0
        while time.monotonic() < deadline:
            if time.monotonic() >= next_hello:
                candidate.write(b"HELLO\r\n")
                next_hello = time.monotonic() + 1
            line = candidate.readline().decode("ascii", errors="replace").strip()
            if line == "READY SERVO_POSE_V1":
                break
            if line.startswith(("ERR", "ERROR:")):
                raise RuntimeError(line)
        else:
            raise TimeoutError("펌웨어 응답이 없습니다. WALK 명령 지원 펌웨어, 포트, baud rate 115200을 확인하세요")
        esp = candidate
        candidate = None
        try:
            capabilities = send_command("CAPS")
        except RuntimeError as exc:
            if "ERR unknown_command" in str(exc):
                raise RuntimeError("연결은 됐지만 현재 ESP32 펌웨어가 보행 명령을 지원하지 않습니다. Arduino IDE에서 esp32-robot-dog-code-master/tools/servoInitPose/servoInitPose.ino를 다시 업로드하세요.") from exc
            raise
        if not capabilities.startswith("CAPS SERVO_POSE_V1") or not {"WALK4", "LEASE1"} <= set(capabilities.split()):
            raise RuntimeError(f"보행 펌웨어 버전이 맞지 않습니다: {capabilities}. WALK4 지원 펌웨어를 업로드하세요.")
        stop_reply = send_command("WALK STOP")
        if stop_reply != "OK WALK STOPPED":
            raise RuntimeError(f"Unexpected stop response: {stop_reply}")
        motion_state_unknown = False
        status = send_command("STATUS")
        try:
            reset_reason = send_command("RESET_REASON").split(maxsplit=1)[1]
        except (RuntimeError, TimeoutError, IndexError):
            reset_reason = "UNKNOWN"
        reset_note = f" 마지막 리셋 원인: {html.escape(reset_reason)}"
        if reset_reason == "BROWNOUT":
            reset_note += " <b>(전원 전압 강하로 ESP32가 재부팅됐습니다. 배터리 충전 상태와 서보 전원을 확인하세요.)</b>"
        port_status.value = f"<b>Connected to {html.escape(PORT)}.</b> {html.escape(status)}{reset_note}"
        update_controls()
        with ui_output:
            clear_output(wait=True)
            print("ESP32 연결 및 펌웨어 응답 확인 완료.")
    except (RuntimeError, ValueError, TypeError, TimeoutError, OSError) as exc:
        if candidate is not None and candidate.is_open:
            candidate.close()
        if esp is not None and esp.is_open:
            esp.close()
        esp = None
        motion_state_unknown = True
        port_status.value = f"<b>Connection failed:</b> {html.escape(str(exc))}"
        connect_button.disabled = False
        update_controls()
        with ui_output:
            clear_output(wait=True)
            print(f"Connection error: {exc}")

def stop_walking():
    global movement_enabled, active_direction, motion_state_unknown
    movement_enabled = False
    active_direction = None
    motion_state_unknown = True
    walk_status.value = "정지 명령 확인 중"
    update_controls()
    if esp is None or not esp.is_open:
        walk_status.value = "정지 확인 불가 - 로봇 상태 불명"
        update_controls()
        raise RuntimeError("연결이 없어 STOP 응답을 확인할 수 없습니다. 로봇 상태가 불명확하면 전원을 차단하세요")
    try:
        reply = send_command("WALK STOP")
        if reply != "OK WALK STOPPED":
            raise RuntimeError(f"Unexpected stop response: {reply}")
    except (RuntimeError, TimeoutError, OSError) as exc:
        walk_status.value = "정지 확인 실패 - 로봇 상태 불명, 전원 차단 필요"
        update_controls()
        raise RuntimeError(f"STOP 응답을 확인하지 못했습니다. 시리얼 연결을 유지합니다: {exc}") from exc
    walk_status.value = "정지 확인됨"
    motion_state_unknown = False
    update_controls()
    return "STOP 응답을 확인했습니다. 서보가 중립 위치에 도달할 때까지 움직일 수 있습니다."

def toggle_direction(direction):
    global active_direction
    if not movement_enabled:
        raise RuntimeError("먼저 움직임을 활성화하세요")
    if active_direction == direction:
        return stop_walking()
    try:
        reply = send_command(f"WALK {direction}", require_motion=True)
        expected = f"OK WALK {direction}"
        if reply != expected:
            raise RuntimeError(f"Unexpected walk response: {reply}")
    except (RuntimeError, TimeoutError, OSError) as exc:
        try:
            stop_walking()
        except (RuntimeError, TimeoutError, OSError) as stop_exc:
            raise RuntimeError(f"보행 명령 응답 오류: {exc}. STOP 확인도 실패했습니다. 전원을 차단하세요: {stop_exc}") from stop_exc
        raise RuntimeError(f"보행 명령 응답 오류로 STOP을 확인했습니다: {exc}") from exc
    active_direction = direction
    start_heartbeat()
    walk_status.value = f"<b>보행 중:</b> {html.escape(direction)} (같은 버튼을 다시 누르면 정지)"
    update_controls()
    return f"{direction} 방향 보행 시작. 같은 방향 버튼을 다시 누르면 정지합니다."

def set_direction_handler(direction):
    def handler(_):
        show_result(lambda: toggle_direction(direction))
    return handler

def connect_and_arm(_):
    global movement_enabled
    def arm():
        global movement_enabled
        if esp is None or not esp.is_open:
            raise RuntimeError("먼저 ESP32에 연결하세요")
        if motion_state_unknown:
            raise RuntimeError("먼저 긴급 정지 응답을 확인하세요")
        movement_enabled = True
        update_controls()
        return "움직임 활성화됨. 로봇을 고정하고 짧게 시험하세요."
    show_result(arm)

def disarm(_):
    global movement_enabled
    def action():
        global movement_enabled
        nonlocal_error = None
        if active_direction is not None:
            try:
                stop_walking()
            except (RuntimeError, TimeoutError, OSError) as exc:
                nonlocal_error = exc
        movement_enabled = False
        update_controls()
        if nonlocal_error is not None:
            raise RuntimeError(f"보행 정지 응답 오류: {nonlocal_error}") from nonlocal_error
        return "움직임을 비활성화했습니다."
    show_result(action)

def disconnect_board(_):
    global esp, movement_enabled, active_direction, motion_state_unknown
    connection = esp
    stop_confirmed = False
    pose_saved = False
    if connection is not None and connection.is_open:
        try:
            stop_walking()
            stop_confirmed = True
        except (RuntimeError, TimeoutError, OSError) as exc:
            port_status.value = f"<b>연결 유지:</b> STOP 확인 실패. 긴급 정지를 다시 시도하거나 전원을 차단하세요. {html.escape(str(exc))}"
            with ui_output:
                clear_output(wait=True)
                print(f"정지 응답을 받지 못해 연결을 유지합니다. 로봇 상태를 확인하고 필요하면 즉시 전원을 차단하세요: {exc}")
            update_controls()
            return
        port_status.value = "서보가 중립 자세에 도착하고 자세가 저장될 때까지 기다리는 중..."
        try:
            pose_saved = wait_for_settled()
        except (RuntimeError, TimeoutError, OSError):
            pose_saved = False
        try:
            connection.close()
        except OSError as exc:
            motion_state_unknown = True
            port_status.value = f"<b>연결 해제 실패:</b> STOP은 확인됐지만 시리얼 포트를 닫지 못했습니다. {html.escape(str(exc))}"
            with ui_output:
                clear_output(wait=True)
                print(f"STOP은 확인됐지만 연결 해제에 실패했습니다: {exc}")
            update_controls()
            return
    esp = None
    movement_enabled = False
    active_direction = None
    motion_state_unknown = True
    walk_status.value = "정지"
    port_status.value = "Not connected. Close Arduino Serial Monitor, then connect the board."
    update_controls()
    with ui_output:
        clear_output(wait=True)
        if not stop_confirmed:
            print("연결을 해제했습니다. 보드의 정지 상태는 확인되지 않았습니다.")
        elif pose_saved:
            print("STOP 확인 후 연결을 해제했습니다. 중립 자세 도착과 저장도 확인했습니다.")
        else:
            print("STOP 확인 후 연결을 해제했습니다. 다만 중립 자세 도착/저장은 확인하지 못했습니다. 다음 접속 때 서보가 빠르게 움직일 수 있습니다.")

def emergency_stop(_):
    show_result(stop_walking)

connect_button.on_click(connect_board)
disconnect_button.on_click(disconnect_board)
arm_button.on_click(connect_and_arm)
disarm_button.on_click(disarm)
stop_button.on_click(emergency_stop)
for direction, button in walk_buttons.items():
    button.on_click(set_direction_handler(direction))
update_controls()
display(widgets.VBox([
    port_status,
    widgets.HBox([connect_button, disconnect_button]),
    widgets.HBox([arm_button, disarm_button]),
    widgets.HTML("Click a direction to start; click the same direction again to stop. Changing direction replaces the current direction."),
    widgets.HBox([walk_buttons["FORWARD"], walk_buttons["BACKWARD"]]),
    widgets.HBox([walk_buttons["LEFT"], walk_buttons["RIGHT"], stop_button]),
    walk_status,
    ui_output,
]))